In [1]:
pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# Optimized Controlled Sentiment Rewriting System
# (Low-memory version, same architecture)
# ============================================================

import re
import torch
from dataclasses import dataclass, field
from typing import Literal

# ── Models ───────────────────────────────────────────────────
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

from sentence_transformers import SentenceTransformer, util
from transformers import pipeline as clf_pipeline

torch.set_grad_enabled(False)

# ─────────────────────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────────────────────

GENERATION_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
SIMILARITY_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SENTIMENT_MODEL  = "distilbert-base-uncased-finetuned-sst-2-english"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ─────────────────────────────────────────────────────────────
# Data Classes
# ─────────────────────────────────────────────────────────────

@dataclass
class RewriteCandidate:
    text: str
    sentiment_label: str
    sentiment_score: float
    similarity_score: float
    rank: int = 0

    def is_correct_sentiment(self, target: str) -> bool:
        return self.sentiment_label.lower() == target.lower()


@dataclass
class RewriteResult:
    original: str
    target_sentiment: str
    candidates: list[RewriteCandidate] = field(default_factory=list)

    @property
    def best(self):
        return self.candidates[0] if self.candidates else None


# ─────────────────────────────────────────────────────────────
# Model Loading (OPTIMIZED)
# ─────────────────────────────────────────────────────────────

def load_generation_model(model_name=GENERATION_MODEL):
    print(f"Loading 4-bit quantized model: {model_name}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    model.eval()
    return tokenizer, model


def load_scoring_models():
    print("Loading scoring models (CPU)...")

    sim_model = SentenceTransformer(SIMILARITY_MODEL, device="cpu")

    sent_clf = clf_pipeline(
        "sentiment-analysis",
        model=SENTIMENT_MODEL,
        device=-1
    )

    return sim_model, sent_clf


# ─────────────────────────────────────────────────────────────
# Prompt
# ─────────────────────────────────────────────────────────────

def build_prompt(sentence: str, target: str, n: int) -> str:
    return (
        f"<s>[INST] Rewrite the sentence with a clearly {target} sentiment "
        f"while preserving meaning. Generate exactly {n} variations.\n\n"
        f"Sentence: \"{sentence}\"\n\nRewrites: [/INST]"
    )


# ─────────────────────────────────────────────────────────────
# Generation (OPTIMIZED)
# ─────────────────────────────────────────────────────────────

def generate_candidates(tokenizer, model, prompt, n):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,              # reduced
            do_sample=True,
            temperature=0.85,
            top_p=0.92,
            num_return_sequences=min(n, 8),  # limit memory
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]

    decoded = [
        tokenizer.decode(o[prompt_len:], skip_special_tokens=True)
        for o in outputs
    ]

    torch.cuda.empty_cache()
    return decoded


# ─────────────────────────────────────────────────────────────
# Parsing + Dedup
# ─────────────────────────────────────────────────────────────

def parse_numbered_list(text):
    lines = text.split("\n")
    results = []

    for line in lines:
        m = re.match(r"^\d+[\.\)]\s+(.+)", line.strip())
        if m:
            results.append(m.group(1).strip())

    return results


def deduplicate(candidates):
    seen = []
    for c in candidates:
        if c not in seen:
            seen.append(c)
    return seen


# ─────────────────────────────────────────────────────────────
# Scoring
# ─────────────────────────────────────────────────────────────

def score_candidates(original, candidates, sim_model, sent_clf, target):
    if not candidates:
        return []

    orig_emb = sim_model.encode(original, convert_to_tensor=True)
    cand_emb = sim_model.encode(candidates, convert_to_tensor=True)

    similarities = util.cos_sim(orig_emb, cand_emb)[0].tolist()
    sentiments = sent_clf(candidates)

    scored = []

    for text, sim, sent in zip(candidates, similarities, sentiments):
        label = sent["label"]
        score = sent["score"]

        scored.append(RewriteCandidate(
            text=text,
            sentiment_label=label,
            sentiment_score=score,
            similarity_score=float(sim)
        ))

    scored.sort(key=lambda c: (
        c.sentiment_label.lower() != target.lower(),
        -c.similarity_score
    ))

    for i, c in enumerate(scored):
        c.rank = i + 1

    return scored


# ─────────────────────────────────────────────────────────────
# Main Pipeline
# ─────────────────────────────────────────────────────────────

class SentimentRewriter:

    def __init__(self):
        self.tokenizer, self.model = load_generation_model()
        self.sim_model, self.sent_clf = load_scoring_models()

    def rewrite(self, sentence, target="positive", n=3):
        sentence = sentence.strip()
        n_raw = n * 2   # reduced oversampling

        prompt = build_prompt(sentence, target, n_raw)

        raw_outputs = generate_candidates(
            self.tokenizer,
            self.model,
            prompt,
            n_raw
        )

        candidates = []
        for r in raw_outputs:
            parsed = parse_numbered_list(r)
            candidates.extend(parsed if parsed else [r.strip()])

        candidates = deduplicate(candidates)

        scored = score_candidates(
            sentence,
            candidates,
            self.sim_model,
            self.sent_clf,
            target
        )

        return RewriteResult(
            original=sentence,
            target_sentiment=target,
            candidates=scored[:n]
        )


# ─────────────────────────────────────────────────────────────
# Example
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    rewriter = SentimentRewriter()

    result = rewriter.rewrite(
        "The food at this restaurant was okay.",
        target="positive",
        n=3
    )

    for c in result.candidates:
        print(f"\n[{c.rank}] {c.text}")
        print(f"Sentiment: {c.sentiment_label} | Sim: {c.similarity_score:.2f}")

Loading 4-bit quantized model: mistralai/Mistral-7B-Instruct-v0.2


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]


[1] The food at this restaurant was acceptable.
Sentiment: POSITIVE | Sim: 0.92

[2] This restaurant's food was acceptable.
Sentiment: POSITIVE | Sim: 0.91

[3] The food at this restaurant was decent.
Sentiment: POSITIVE | Sim: 0.88


In [3]:
TEST_CASES = [
    # Neutral → Positive
    {"sentence": "The food was average.", "target": "positive"},
    {"sentence": "The experience was okay overall.", "target": "positive"},

    # Neutral → Negative
    {"sentence": "The service was fine.", "target": "negative"},
    {"sentence": "The product works as expected.", "target": "negative"},

    # Strong Positive → Negative
    {"sentence": "I absolutely loved this movie.", "target": "negative"},
    {"sentence": "This is the best purchase I've ever made.", "target": "negative"},

    # Strong Negative → Positive
    {"sentence": "The app is terrible and keeps crashing.", "target": "positive"},
    {"sentence": "I hate how slow this service is.", "target": "positive"},

    # Mixed sentiment
    {"sentence": "The room was clean but the staff was rude.", "target": "positive"},
    {"sentence": "The design is nice but performance is poor.", "target": "negative"},

    # Longer sentence
    {"sentence": "The hotel location was convenient, but the amenities were lacking and the service could have been better.", "target": "positive"},
]

In [4]:
def run_manual_tests(rewriter, test_cases, n=3):
    for i, tc in enumerate(test_cases, 1):
        print(f"\n{'='*70}")
        print(f"Test {i}")
        print(f"Original : {tc['sentence']}")
        print(f"Target   : {tc['target']}")
        print(f"{'='*70}")

        result = rewriter.rewrite(
            tc["sentence"],
            target=tc["target"],
            n=n
        )

        for c in result.candidates:
            status = "✓" if c.is_correct_sentiment(tc["target"]) else "✗"
            print(f"\n[{c.rank}] {status}")
            print(f"Text       : {c.text}")
            print(f"Sentiment  : {c.sentiment_label} ({c.sentiment_score:.2f})")
            print(f"Similarity : {c.similarity_score:.2f}")

In [5]:
rewriter = SentimentRewriter()
run_manual_tests(rewriter, TEST_CASES, n=3)

Loading 4-bit quantized model: mistralai/Mistral-7B-Instruct-v0.2


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading scoring models (CPU)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Test 1
Original : The food was average.
Target   : positive

[1] ✓
Text       : The food was solid and appetizing.
Sentiment  : POSITIVE (1.00)
Similarity : 0.75

[2] ✓
Text       : The food was satisfactory.
Sentiment  : POSITIVE (1.00)
Similarity : 0.74

[3] ✓
Text       : The food was quite acceptable.
Sentiment  : POSITIVE (1.00)
Similarity : 0.72

Test 2
Original : The experience was okay overall.
Target   : positive

[1] ✓
Text       : "The experience was satisfactory."
Sentiment  : POSITIVE (1.00)
Similarity : 0.66

[2] ✓
Text       : "The experience was generally satisfactory."
Sentiment  : POSITIVE (1.00)
Similarity : 0.66

[3] ✓
Text       : "The experience was not bad, it was quite enjoyable."
Sentiment  : POSITIVE (1.00)
Similarity : 0.63

Test 3
Original : The service was fine.
Target   : negative

[1] ✓
Text       : The service was unremarkable.
Sentiment  : NEGATIVE (1.00)
Similarity : 0.71

[2] ✓
Text       : The service was unsatisfactory.
Sentiment  : NEGATIVE (1.00)

**EVALUATION**

In [6]:
import pandas as pd

def load_yelp_dataset(path):
    df = pd.read_csv(path, sep="\t")

    # Convert "4star" → 4
    df["rating_num"] = df["rating"].str[0].astype(int)

    # Sentiment label
    df["label"] = df["rating_num"].apply(lambda x: 1 if x >= 3 else 0)

    # Flip for rewriting task
    df["target"] = df["label"].apply(
        lambda x: "negative" if x == 1 else "positive"
    )

    df.rename(columns={"content": "text"}, inplace=True)

    return df[["text", "label", "target"]]

In [7]:
def evaluate_yelp_basic(rewriter, df, samples=20, n=3):
    df = df.sample(samples).reset_index(drop=True)

    total = 0
    top1_correct = 0
    topk_correct = 0
    sim_total = 0

    originals = []
    best_outputs = []

    for i, row in df.iterrows():
        print(f"\n[{i+1}] Input: {row['text'][:80]}...")
        print(f"Target: {row['target']}")

        result = rewriter.rewrite(
            row["text"],
            target=row["target"],
            n=n
        )

        if not result.candidates:
            continue

        best = result.candidates[0]

        originals.append(row["text"])
        best_outputs.append(best.text)

        total += 1

        # Top-1 Accuracy
        if best.is_correct_sentiment(row["target"]):
            top1_correct += 1

        # Top-K Success
        if any(c.is_correct_sentiment(row["target"]) for c in result.candidates):
            topk_correct += 1

        # Similarity
        sim_total += best.similarity_score

        # Print outputs
        for c in result.candidates:
            status = "✓" if c.is_correct_sentiment(row["target"]) else "✗"
            print(f"\n[{c.rank}] {status}")
            print(f"Text       : {c.text}")
            print(f"Sentiment  : {c.sentiment_label} ({c.sentiment_score:.2f})")
            print(f"Similarity : {c.similarity_score:.2f}")

    print("\n===== BASIC METRICS =====")
    print(f"Top-1 Accuracy : {top1_correct/total:.3f}")
    print(f"Top-{n} Success: {topk_correct/total:.3f}")
    print(f"Avg Similarity : {sim_total/total:.3f}")

    return originals, best_outputs

In [8]:
!pip -q install bert-score evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


In [9]:
from bert_score import score as bertscore
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

def compute_advanced_metrics(originals, outputs):

    P, R, F1 = bertscore(outputs, originals, lang="en", verbose=False)

    bleu_score = bleu.compute(
        predictions=outputs,
        references=[[o] for o in originals]
    )["bleu"]

    rouge_score = rouge.compute(
        predictions=outputs,
        references=originals
    )["rougeL"]

    meteor_score = meteor.compute(
        predictions=outputs,
        references=originals
    )["meteor"]

    print("\n===== ADVANCED METRICS =====")
    print(f"BERTScore : {F1.mean().item():.3f}")
    print(f"BLEU      : {bleu_score:.3f}")
    print(f"ROUGE-L   : {rouge_score:.3f}")
    print(f"METEOR    : {meteor_score:.3f}")

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


In [10]:
import pandas as pd

path = "/kaggle/input/datasets/jayathilagaramajayam/yelp-review-dataset/test.tsv"

df = pd.read_csv(path, sep="\t")

print("Loaded successfully")

Loaded successfully


In [11]:
print(df.columns)
print(df.head())

Index(['content'], dtype='object')
                                             content
0  Bette than the average take out but not yout s...
1  Pizza  slice is not a friend to my palate. The...
2  I probably get to dine in fancy-pansy restaura...
3  Made reservations for 7PM on Friday and showed...
4  Upstate has been on my list for years - so hap...


In [12]:
def load_yelp_dataset(path, sentiment_model):
    df = pd.read_csv(path, sep="\t")

    # Rename column
    df.rename(columns={"content": "text"}, inplace=True)

    print("Generating sentiment labels (safe mode)...")

    texts = df["text"].tolist()

    # 🔥 FIX: truncate long texts
    texts = [t[:1000] for t in texts]   # simple char-level truncation

    preds = sentiment_model(
        texts,
        truncation=True,        # ✅ critical fix
        max_length=512          # ✅ ensures no overflow
    )

    df["label"] = [
        1 if p["label"] == "POSITIVE" else 0
        for p in preds
    ]

    # Flip for rewriting
    df["target"] = df["label"].apply(
        lambda x: "negative" if x == 1 else "positive"
    )

    return df[["text", "label", "target"]]

In [13]:
df = load_yelp_dataset(
    "/kaggle/input/datasets/jayathilagaramajayam/yelp-review-dataset/test.tsv",
    rewriter.sent_clf
)

originals, outputs = evaluate_yelp_basic(
    rewriter,
    df,
    samples=20,
    n=3
)

compute_advanced_metrics(originals, outputs)

Generating sentiment labels (safe mode)...

[1] Input: Ordered the Wild Spirit smoothie with strawberry, banana, and almond milk. I wat...
Target: positive

[1] ✓
Text       : I decided to try the Wild Spirit smoothie with strawberries, banana, and almond milk. To my surprise, the staff member added the whole strawberries, with stems attached, into the blender. I was curious about this unusual method and inquired why they didn't remove the stems. He explained that these strawberries are pre-cleaned and specifically used for making this smoothie. Although I had never seen this before, I trusted his expertise and took a sip. To my delight, the smoothie was incredibly delicious and full of flavor.
Sentiment  : POSITIVE (1.00)
Similarity : 0.93

[2] ✓
Text       : I ordered a Wild Spirit smoothie with strawberries, banana, and almond milk. To my surprise, the staff member added the fresh strawberries with their stems into the blender. He assured me they were pre-cleaned and this was the tr

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



===== ADVANCED METRICS =====
BERTScore : 0.891
BLEU      : 0.078
ROUGE-L   : 0.327
METEOR    : 0.322
